# Baseline Retriever for Applied Behavior Analysis (ABA)

### Download Dependencies

In [1]:
#Download dependencies

import nltk

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/VyasSrinivasan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/VyasSrinivasan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Import Libraries

In [2]:
#Import libraries

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from pathlib import Path
import string
import re
import joblib
import json
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pickle
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, Dense, Flatten, Conv1D, MaxPooling1D, SimpleRNN, GRU, LSTM, LSTM, Input, Embedding, TimeDistributed, Flatten, Dropout,Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [3]:
inputFile = "../data/abaDatasetV1.csv"

In [4]:
df = pd.read_csv(inputFile)
df.head()

,ID,Antecedent,Behavior,Consequence,Emotion_Tag,Context
0,1,My friend ignored me at lunch,I felt sad and didn’t talk to anyone,It’s okay to feel left out. Try checking in wi...,Sadness,Social
1,2,The teacher raised their voice at me,I froze and couldn’t speak,It’s normal to feel nervous when someone uses ...,Anxiety,School
2,3,I made a mistake on my homework,I felt embarrassed and threw it away,Mistakes happen to everyone. You can always fi...,Guilt,School
3,4,Someone cut me in line,I shouted at them,Anger can be hard to control. Counting to five...,Anger,Public
4,5,My classmate complimented my artwork,I smiled and said thank you,That’s a great example of positive reinforceme...,Happiness,School


In [5]:
df.keys()

Index(['ID', 'Antecedent', 'Behavior', 'Consequence', 'Emotion_Tag',
       'Context'],
      dtype='object')

In [6]:
df

,ID,Antecedent,Behavior,Consequence,Emotion_Tag,Context
0,1,My friend ignored me at lunch,I felt sad and didn’t talk to anyone,It’s okay to feel left out. Try checking in wi...,Sadness,Social
1,2,The teacher raised their voice at me,I froze and couldn’t speak,It’s normal to feel nervous when someone uses ...,Anxiety,School
2,3,I made a mistake on my homework,I felt embarrassed and threw it away,Mistakes happen to everyone. You can always fi...,Guilt,School
3,4,Someone cut me in line,I shouted at them,Anger can be hard to control. Counting to five...,Anger,Public
4,5,My classmate complimented my artwork,I smiled and said thank you,That’s a great example of positive reinforceme...,Happiness,School
...,...,...,...,...,...,...
115,116,Someone enjoyed a joke I made,I felt happy and confident,Positive social reactions help reinforce comfo...,Joy,Social
116,117,Someone complimented my outfit,I felt confident,Small compliments boost self-esteem — it’s oka...,Confidence,Social
117,118,My friend checked in on me without me asking,I felt cared for,Being cared for strengthens connection — it’s ...,Care,Relationships
118,119,A coworker said my idea was creative,I felt proud,Creativity being recognized encourages more in...,Pride,Work


In [7]:
texts = df["Antecedent"].fillna("") + " " + df["Behavior"].fillna("") + " " + df["Consequence"].fillna("")

In [8]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

In [9]:
def get_best_match(user_input, min_sim=0.2):
    user_vec = vectorizer.transform([user_input])
    sims = cosine_similarity(user_vec, X).flatten()
    idx = sims.argmax()
    best_sim = sims[idx]
    best_row = df.iloc[idx]
    return best_row, best_sim

In [10]:
def main():
    print("ABA Chatbot — Baseline")
    print("Type 'quit' to exit.\n")

    while True:
        text = input("You: ")
        if text.lower() == "quit":
            break

        row, sim = get_best_match(text)

        if sim < 0.2:
            print("\nAI: I’m still learning and I’m not sure I fully understood this situation.")
            print("    But it sounds important. It might help to talk to someone you trust about how you feel.\n")
        else:
            print("\nAI:", row["Consequence"])
            print("(Recognized emotion:", row["Emotion_Tag"] + " — similarity:", round(sim, 3), ")\n")


if __name__ == "__main__":
    main()

ABA Chatbot — Baseline
Type 'quit' to exit.

You: Someone told me that the recent podcast episodes I made are stale and to suspend the podcast. It made me feel discouraged

AI: It’s normal to feel hurt by criticism, but one comment doesn’t define your creativity or potential.
(Recognized emotion: Sadness — similarity: 0.287 )

You: My friend sent me a text saying "You are a good friend" which made me feel seen and valued.

AI: Feeling appreciated reinforces healthy relationships — enjoy the affirmation.
(Recognized emotion: Appreciation — similarity: 0.482 )

You: My friend told me that he is grateful to have someone as cool and awesome as me in his life.

AI: Belief from others can fuel motivation — take it as momentum.
(Recognized emotion: Hope — similarity: 0.315 )

You: QUIT
